In [1]:
# Phase 2 — Database Modeling
# Project: Savoria Restaurant Group — Financial Analysis
# Author: Leo Sanchez

# ── Libraries ──────────────────────────────────────────
import sqlite3
import pandas as pd

# Load clean data from Phase 1
df = pd.read_csv('data_clean.csv', parse_dates=['date'])

# Connect to database
conn = sqlite3.connect('savoria.db')
cursor = conn.cursor()

print("Libraries loaded ✓")
print(f"Clean data loaded: {len(df):,} records")
print(f"Database connected: savoria.db")

Libraries loaded ✓
Clean data loaded: 27,127 records
Database connected: savoria.db


In [2]:
# ── Create Database Tables ──────────────────────────────
cursor.executescript("""
    DROP TABLE IF EXISTS fact_sales;
    DROP TABLE IF EXISTS dim_branch;
    DROP TABLE IF EXISTS dim_category;
    DROP TABLE IF EXISTS dim_date;
""")

# Dimension table — Branches
cursor.execute("""
    CREATE TABLE dim_branch (
        branch_id    TEXT PRIMARY KEY,
        branch_name  TEXT NOT NULL,
        city         TEXT NOT NULL,
        size         TEXT NOT NULL
    )
""")

# Dimension table — Menu Categories
cursor.execute("""
    CREATE TABLE dim_category (
        category_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        category_name TEXT NOT NULL,
        avg_price     REAL NOT NULL,
        food_cost_pct REAL NOT NULL
    )
""")

# Dimension table — Date
cursor.execute("""
    CREATE TABLE dim_date (
        date_id    TEXT PRIMARY KEY,
        month      INTEGER NOT NULL,
        quarter    INTEGER NOT NULL,
        year       INTEGER NOT NULL,
        is_weekend INTEGER NOT NULL
    )
""")

# Fact table — Sales
cursor.execute("""
    CREATE TABLE fact_sales (
        sale_id      INTEGER PRIMARY KEY AUTOINCREMENT,
        date_id      TEXT NOT NULL,
        branch_id    TEXT NOT NULL,
        category_id  INTEGER NOT NULL,
        covers       INTEGER NOT NULL,
        orders       INTEGER NOT NULL,
        revenue      REAL NOT NULL,
        food_cost    REAL NOT NULL,
        gross_profit REAL NOT NULL,
        gross_margin REAL NOT NULL,
        is_outlier   INTEGER NOT NULL,
        FOREIGN KEY (date_id)     REFERENCES dim_date(date_id),
        FOREIGN KEY (branch_id)   REFERENCES dim_branch(branch_id),
        FOREIGN KEY (category_id) REFERENCES dim_category(category_id)
    )
""")

conn.commit()
print("4 tables created ✓")
print("\nSchema:")
print("  dim_branch    — branch dimensions")
print("  dim_category  — menu category dimensions")
print("  dim_date      — date dimensions")
print("  fact_sales    — central fact table")

4 tables created ✓

Schema:
  dim_branch    — branch dimensions
  dim_category  — menu category dimensions
  dim_date      — date dimensions
  fact_sales    — central fact table


In [4]:
# ── Business Structure (redefined for this notebook) ────
branches = {
    'BR001': {'name': 'Downtown',    'city': 'Mexico City',   'opened': '2020-01-15', 'size': 'large'},
    'BR002': {'name': 'Polanco',     'city': 'Mexico City',   'opened': '2020-06-01', 'size': 'large'},
    'BR003': {'name': 'Guadalajara', 'city': 'Guadalajara',   'opened': '2021-03-10', 'size': 'medium'},
    'BR004': {'name': 'Monterrey',   'city': 'Monterrey',     'opened': '2021-09-20', 'size': 'medium'},
    'BR005': {'name': 'Reforma',     'city': 'Mexico City',   'opened': '2022-02-14', 'size': 'small'},
}

menu_categories = {
    'Starters':    {'avg_price': 85,  'food_cost_pct': 0.28},
    'Main Course': {'avg_price': 180, 'food_cost_pct': 0.32},
    'Desserts':    {'avg_price': 75,  'food_cost_pct': 0.25},
    'Beverages':   {'avg_price': 60,  'food_cost_pct': 0.18},
    'Alcohol':     {'avg_price': 120, 'food_cost_pct': 0.22},
}

print("Business structure loaded ✓")

Business structure loaded ✓


In [5]:
# ── Populate Dimension Tables ───────────────────────────

# dim_branch
branch_data = pd.DataFrame([
    {'branch_id': bid, 'branch_name': info['name'],
     'city': info['city'], 'size': info['size']}
    for bid, info in branches.items()
])
branch_data.to_sql('dim_branch', conn, if_exists='append', index=False)
print(f"dim_branch populated:   {len(branch_data)} rows")

# dim_category
category_data = pd.DataFrame([
    {'category_name': cat, 'avg_price': info['avg_price'],
     'food_cost_pct': info['food_cost_pct']}
    for cat, info in menu_categories.items()
])
category_data.to_sql('dim_category', conn, if_exists='append', index=False)
print(f"dim_category populated: {len(category_data)} rows")

# dim_date
date_data = df[['date', 'month', 'quarter', 'year', 'is_weekend']].drop_duplicates(subset='date')
date_data = date_data.rename(columns={'date': 'date_id'})
date_data['date_id'] = date_data['date_id'].astype(str).str[:10]
date_data['is_weekend'] = date_data['is_weekend'].astype(int)
date_data.to_sql('dim_date', conn, if_exists='append', index=False)
print(f"dim_date populated:     {len(date_data)} rows")

conn.commit()
print("\nDimension tables ready ✓")

dim_branch populated:   5 rows
dim_category populated: 5 rows
dim_date populated:     1096 rows

Dimension tables ready ✓


In [6]:
# ── Populate Fact Table ─────────────────────────────────

# Get category IDs from dim_category
cat_ids = pd.read_sql("SELECT category_id, category_name FROM dim_category", conn)
cat_map = dict(zip(cat_ids['category_name'], cat_ids['category_id']))

# Prepare fact table
fact = df.copy()
fact['date_id']     = fact['date'].astype(str).str[:10]
fact['category_id'] = fact['category'].map(cat_map)
fact['is_outlier']  = fact['is_outlier'].astype(int)

# Select only fact table columns
fact_cols = ['date_id', 'branch_id', 'category_id', 'covers',
             'orders', 'revenue', 'food_cost', 'gross_profit',
             'gross_margin', 'is_outlier']

fact[fact_cols].to_sql('fact_sales', conn, if_exists='append', index=False)
conn.commit()

# Validate
count = pd.read_sql("SELECT COUNT(*) as total FROM fact_sales", conn)
print(f"fact_sales populated: {count['total'].values[0]:,} rows ✓")

fact_sales populated: 27,127 rows ✓


In [7]:
# ── Analytical Queries ──────────────────────────────────

# Query 1 — Revenue and gross profit by branch
q1 = """
SELECT 
    b.branch_name,
    b.city,
    b.size,
    COUNT(*)                          AS total_records,
    ROUND(SUM(f.revenue), 0)          AS total_revenue,
    ROUND(SUM(f.gross_profit), 0)     AS total_gross_profit,
    ROUND(AVG(f.gross_margin) * 100, 1) AS avg_gross_margin_pct,
    ROUND(SUM(f.revenue) / 
          COUNT(DISTINCT f.date_id), 0) AS avg_daily_revenue
FROM fact_sales f
JOIN dim_branch b ON f.branch_id = b.branch_id
GROUP BY b.branch_name, b.city, b.size
ORDER BY total_revenue DESC
"""

result_q1 = pd.read_sql(q1, conn)
print("=== REVENUE BY BRANCH ===")
print(result_q1.to_string(index=False))

=== REVENUE BY BRANCH ===
branch_name        city   size  total_records  total_revenue  total_gross_profit  avg_gross_margin_pct  avg_daily_revenue
    Polanco Mexico City  large           5468     30243745.0          21963528.0                  75.0            27595.0
   Downtown Mexico City  large           5467     30211182.0          21944212.0                  75.0            27565.0
Guadalajara Guadalajara medium           5466     20095430.0          14657328.0                  75.1            18335.0
  Monterrey   Monterrey medium           5470     20089501.0          14639103.0                  75.0            18330.0
    Reforma Mexico City  small           5256     12142522.0           8863369.0                  75.1            11542.0


In [8]:
# Query 2 — Revenue by category
q2 = """
SELECT
    c.category_name,
    ROUND(SUM(f.revenue), 0)            AS total_revenue,
    ROUND(SUM(f.gross_profit), 0)       AS total_gross_profit,
    ROUND(AVG(f.gross_margin) * 100, 1) AS avg_gross_margin_pct,
    ROUND(SUM(f.revenue) * 100.0 /
          (SELECT SUM(revenue) FROM fact_sales), 1) AS revenue_share_pct
FROM fact_sales f
JOIN dim_category c ON f.category_id = c.category_id
GROUP BY c.category_name
ORDER BY total_revenue DESC
"""

result_q2 = pd.read_sql(q2, conn)
print("=== REVENUE BY MENU CATEGORY ===")
print(result_q2.to_string(index=False))

=== REVENUE BY MENU CATEGORY ===
category_name  total_revenue  total_gross_profit  avg_gross_margin_pct  revenue_share_pct
  Main Course     53663603.0          36380039.0                  67.9               47.6
      Alcohol     17053687.0          13341438.0                  78.0               15.1
    Beverages     16315135.0          13412940.0                  82.0               14.5
     Starters     16248492.0          11767690.0                  72.1               14.4
     Desserts      9501462.0           7165433.0                  75.1                8.4


In [9]:
# Query 3 — Monthly revenue trend by year
q3 = """
SELECT
    d.year,
    d.month,
    ROUND(SUM(f.revenue), 0)          AS monthly_revenue,
    ROUND(SUM(f.gross_profit), 0)     AS monthly_gross_profit,
    ROUND(AVG(f.gross_margin) * 100, 1) AS avg_margin_pct,
    COUNT(DISTINCT f.date_id)         AS operating_days
FROM fact_sales f
JOIN dim_date d ON f.date_id = d.date_id
GROUP BY d.year, d.month
ORDER BY d.year, d.month
"""

result_q3 = pd.read_sql(q3, conn)
print("=== MONTHLY REVENUE TREND ===")
print(result_q3.to_string(index=False))

=== MONTHLY REVENUE TREND ===
 year  month  monthly_revenue  monthly_gross_profit  avg_margin_pct  operating_days
 2022      1        2222996.0             1618501.0            75.1              31
 2022      2        2000984.0             1461234.0            75.2              28
 2022      3        2598766.0             1901239.0            75.2              31
 2022      4        2717809.0             1989229.0            75.2              30
 2022      5        2898762.0             2112880.0            75.1              31
 2022      6        2932403.0             2146640.0            75.2              30
 2022      7        3319458.0             2427134.0            75.1              31
 2022      8        3013654.0             2194142.0            75.0              31
 2022      9        2699684.0             1965564.0            75.0              30
 2022     10        3006564.0             2191614.0            75.1              31
 2022     11        2938980.0             2143

In [10]:
# Query 4 — Weekend vs Weekday performance
q4 = """
SELECT
    b.branch_name,
    CASE WHEN d.is_weekend = 1 THEN 'Weekend' ELSE 'Weekday' END AS day_type,
    COUNT(DISTINCT f.date_id)           AS total_days,
    ROUND(SUM(f.revenue), 0)            AS total_revenue,
    ROUND(SUM(f.revenue) /
          COUNT(DISTINCT f.date_id), 0) AS avg_daily_revenue,
    ROUND(AVG(f.covers), 0)            AS avg_covers
FROM fact_sales f
JOIN dim_branch b ON f.branch_id  = b.branch_id
JOIN dim_date   d ON f.date_id    = d.date_id
GROUP BY b.branch_name, day_type
ORDER BY b.branch_name, day_type DESC
"""

result_q4 = pd.read_sql(q4, conn)
print("=== WEEKEND VS WEEKDAY PERFORMANCE ===")
print(result_q4.to_string(index=False))

=== WEEKEND VS WEEKDAY PERFORMANCE ===
branch_name day_type  total_days  total_revenue  avg_daily_revenue  avg_covers
   Downtown  Weekend         314     10744455.0            34218.0        99.0
   Downtown  Weekday         782     19466726.0            24894.0        70.0
Guadalajara  Weekend         314      7244570.0            23072.0        65.0
Guadalajara  Weekday         782     12850859.0            16433.0        47.0
  Monterrey  Weekend         314      7327142.0            23335.0        65.0
  Monterrey  Weekday         782     12762359.0            16320.0        46.0
    Polanco  Weekend         314     10876329.0            34638.0       100.0
    Polanco  Weekday         782     19367416.0            24767.0        70.0
    Reforma  Weekend         300      4340903.0            14470.0        41.0
    Reforma  Weekday         752      7801619.0            10374.0        29.0


## Phase 2 — Database Modeling

### Schema — Star Schema
- `dim_branch`   — 5 branches across Mexico
- `dim_category` — 5 menu categories
- `dim_date`     — 1,096 unique dates (2022-2024)
- `fact_sales`   — 27,127 transaction records

### Key Queries & Findings

**By Branch**
- Polanco and Downtown lead with ~$30M revenue each
- Consistent 75% gross margin across all branches — scalable model

**By Menu Category**
- Main Course drives 47.6% of revenue
- Beverages and Alcohol have the highest margins (78-82%)
- Strategic opportunity: increase beverage mix to improve overall margin

**Monthly Trend**
- Clear seasonality: Q4 strongest, Q1 weakest
- Consistent year-over-year growth across all branches

**Weekend vs Weekday**
- Weekends generate ~30% more daily revenue than weekdays
- Opportunity: develop weekday traffic programs

### Output Files
- `savoria.db` — SQLite database with Star Schema